In [1]:
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm

# Import RAG

In [2]:
import sys
sys.path.append('../scripts')
import rag
import vectors

# Load synthetic data

In [3]:
df_synth = pd.read_csv('../data/data-synth-question.csv', sep='\t', dtype=str)
df_synth

,pmid,ollama_seed,synthetic_question
0,40247149,0,What role do computer algorithms play in devel...
1,40247149,1,Is it more efficient to use machine learning a...
2,40247149,2,What role do researchers believe software tool...
3,40247149,3,What role do expert systems play in improving ...
4,40247149,4,What is one potential limitation of using fMRI...
...,...,...,...
495,40933682,0,What is a major challenge that autistic studen...
496,40933682,1,Does a lack of visibility and understanding fr...
497,40933682,2,What drives college students with autism to de...
498,40933682,3,What is the primary reason that autistic colle...


# Demo retriever

In [4]:
# select question to demonstrate
demo_query = df_synth.iloc[0]['synthetic_question']
print(demo_query)

What role do computer algorithms play in developing effective fMRI features for diagnosing autism spectrum disorders?


In [5]:
# demonstrate vector search
rag.vsearch(demo_query, num_results=2)

[{'pmid': '40563741',
  'elocationid': 'pii: 569',
  'title': 'An Autism Spectrum Disorder Identification Method Based on 3D-CNN and Segmented Temporal Decision Network.',
  'journal': 'Brain sciences',
  'year': '2025',
  'author': 'Zhiling Liu, Ye Chen, Xinrui Dong, Jing Liu',
  'affiliation': 'Faculty of Psychology, Beijing Normal University, Beijing 100875, China. College of Computer Science, Beijing University of Technology, Beijing 100124, China.',
  'abstract': "(1) Background: Autism Spectrum Disorder (ASD) is a neurodevelopmental disorder characterized by social communication deficits and repetitive behaviors. Functional MRI (fMRI) has been widely applied to investigate brain functional abnormalities associated with ASD, yet challenges remain due to complex data characteristics and limited spatiotemporal information capture. This study aims to improve the ability to capture spatiotemporal dynamics of brain activity by proposing an advanced framework. (2) Methods: This study pr

In [6]:
# demonstrate keyword search
rag.kwsearch(demo_query, num_results=2)

[{'pmid': '40821657',
  'elocationid': 'pii: 102489',
  'title': 'Functional upper-extremity movements in autism: A narrative literature review.',
  'journal': 'Research in autism spectrum disorders',
  'year': '2024',
  'author': 'Shanan Sun, Nicholas E Fears, Haylie L Miller',
  'affiliation': 'School of Kinesiology, University of Michigan, Ann Arbor, MI, USA.',
  'abstract': 'Many autistic individuals exhibit clinically-significant motor difficulties. Previous reviews focused on overall motor ability or coordination, but with little attention paid to quantifying differences in upper extremity skills, which are critical to many activities of daily living. Our objective was to identify and evaluate the published literature on upper extremity motor skills of autistic people.'},
 {'pmid': '40881177',
  'elocationid': 'pii: 102460',
  'title': 'Exploring the effects of age and sex on sensory sensitivities in middle and older aged autistic adults.',
  'journal': 'Research in autism spectr

# Evaluate retriever, by hit rate and mean reciprocal rank (MRR)

## Define functions

In [7]:
def get_relevance_total(synth_records, search_function, num_results):
    # initialize relevance total
    relevance_total = []
    # for each document, make relevance
    for record in tqdm(synth_records):
        query = record['synthetic_question']
        true_pmid = record['pmid']
        search_results = search_function(query, num_results)
        relevance = [true_pmid==d['pmid'] for d in search_results]
        relevance_total.append(relevance)
    return relevance_total

In [8]:
def hit_rate(relevance_total):
    hit_boolean = [True in line for line in relevance_total]
    numerator = sum(hit_boolean)
    denominator = len(hit_boolean)
    return numerator/denominator

In [9]:
def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1/(rank+1)
    return total_score/len(relevance_total)

In [10]:
def get_questions_by_correctness(synth_records, relevance_total):
    questions_answered_correct = []
    questions_answered_incorrect = []
    for i in range(len(synth_records)):
        relevance = relevance_total[i]
        question = synth_records[i]['synthetic_question']
        if True in relevance:
            questions_answered_correct.append(question)
        else:
            questions_answered_incorrect.append(question)
    all_questions = {'correct' : questions_answered_correct, \
    'incorrect' : questions_answered_incorrect}
    return all_questions

## Work

In [11]:
# cast as list of dictionaries
synth_records = df_synth.to_dict('records')
print(len(synth_records))

500


In [12]:
# number of documents retrieved
num_results = rag.config['num_results']
print(num_results)

5


In [13]:
# for vector search, get relevance total
print(datetime.now())
relevance_total_v = get_relevance_total(synth_records=synth_records, \
search_function=rag.vsearch, num_results=num_results)
print(datetime.now())

2025-09-12 22:32:04.661414


  0%|          | 0/500 [00:00<?, ?it/s]

2025-09-12 22:33:06.591137


In [14]:
(lambda x : {'hit rate' : hit_rate(x), 'mrr' : mrr(x)})(relevance_total_v)

{'hit rate': 0.438, 'mrr': 0.3631333333333333}

In [15]:
# for vector search, see questions correctly and incorrectly answered
qbc_v = get_questions_by_correctness(synth_records, relevance_total_v)
print('---correct---')
print('\n'.join(qbc_v['correct'][:10]))
print()
print('---incorrect---')
print('\n'.join(qbc_v['incorrect'][:10]))

---correct---
What role do computer algorithms play in developing effective fMRI features for diagnosing autism spectrum disorders?
Is it more efficient to use machine learning algorithms to learn the underlying patterns in fMRI data rather than relying solely on traditional statistical analysis methods?
What role do researchers believe software tools play in shaping the accuracy of fMRI-based brain disease analysis?
What role do expert systems play in improving the accuracy of fMRI data analysis?
How do cognitive dysmetria and functional connectivity relate to symptoms and traits in individuals with schizophrenia?
Is fragile X syndrome caused by DNA methylation of the FMR1 gene promoter?
What triggers the abnormal methylation of the fragile X syndrome gene in people with autism spectrum disorder?
What is the underlying cause of the difference in methylation status between Fragile X syndrome and its genetic engineering counterparts?
What role do immune cells play in the development or 

In [16]:
# for keyword search, get relevance total
print(datetime.now())
relevance_total_kw = get_relevance_total(synth_records=synth_records, \
search_function=rag.kwsearch, num_results=num_results)
print(datetime.now())

2025-09-12 22:33:06.627200


  0%|          | 0/500 [00:00<?, ?it/s]

2025-09-12 22:33:08.949744


In [17]:
(lambda x : {'hit rate' : hit_rate(x), 'mrr' : mrr(x)})(relevance_total_kw)

{'hit rate': 0.442, 'mrr': 0.3727333333333334}

In [18]:
# for keyword search, seee questions correctly and incorrectly answered
qbc_kw = get_questions_by_correctness(synth_records, relevance_total_kw)
print('---correct---')
print('\n'.join(qbc_kw['correct'][:10]))
print()
print('---incorrect---')
print('\n'.join(qbc_kw['incorrect'][:10]))

---correct---
What role do researchers believe software tools play in shaping the accuracy of fMRI-based brain disease analysis?
How do cognitive dysmetria and functional connectivity relate to symptoms and traits in individuals with schizophrenia?
Is fragile X syndrome caused by DNA methylation of the FMR1 gene promoter?
What triggers the abnormal methylation of the fragile X syndrome gene in people with autism spectrum disorder?
What is the underlying cause of the difference in methylation status between Fragile X syndrome and its genetic engineering counterparts?
What role do immune cells play in the development or progression of autism?
How do immune responses in individuals with autism affect brain development?
What is the role of immune cells in regulating inflammation and tissue repair in individuals with autism?
What is the role of immune cells in regulating inflammation within the brain?
What is the most effective treatment approach for autistic adults experiencing catatonic s

In [19]:
print(datetime.now())

2025-09-12 22:33:08.992828
